# Lecture 8 — Choropleth Maps

> **Dataset:** World Development Indicators (World Bank) + Gapminder (built-in)



---
## Opening: Model Answer Review 

---
## Choropleth Maps


### What is a Choropleth?

The word *Choropleth* comes from two Greek words: *Choros* (area) and *Plethos* (multitude).

Choropleth maps use shaded or patterned areas proportional to a statistical variable — they encode a value by colouring geographic regions.

> 💡 **A choropleth earns its complexity when the geographic pattern itself is the insight — when WHERE something is happening is as important as WHAT is happening**

**Use a choropleth when:**
- The geographic distribution IS the story (e.g. development is concentrated in specific regions)
- Your audience needs spatial context to understand the data
- Adjacent regions are meaningfully comparable

**Use a bar chart instead when:**
- You want to show ranking (bars sort; maps don't)
- The pattern is random geographically
- Precise comparison between specific countries matters more than pattern

**Choropleth design rules:**
- **Sequential scale** (single hue, light to dark) for one-directional data (poverty rate, life expectancy)
- **Diverging scale** (two hues, white/neutral midpoint) for above/below average or +/- data
- Always include a colour legend
- Zoom in to the relevant region if showing a subset


### Two inputs required

Every choropleth needs:

1. **Geometric information** — the shapes of the regions, either:
   - Plotly's built-in geometries (world countries via ISO-3 codes, or US states)
   - A custom **GeoJSON file** for any other geography

2. **A value column** — what to shade each region by

### Key `px.choropleth` arguments

| Argument | Purpose |
|---|---|
| `locations` | The column that identifies each region |
| `locationmode` | How to interpret locations: `'ISO-3'`, `'country names'`, or `'USA-states'` |
| `geojson` | Custom GeoJSON file (when not using built-in geometries) |
| `featureidkey` | Which property in the GeoJSON matches your `locations` column |
| `color` | The column to shade by |
| `scope` | Zoom to a continent: `'world'`, `'europe'`, `'africa'`, `'asia'`, `'north america'`, `'south america'`, `'usa'` |
| `fitbounds` | `'locations'` to auto-zoom to your data |
| `color_continuous_scale` | Sequential or diverging colour scale |
| `color_continuous_midpoint` | Set the neutral midpoint for diverging scales |


---
## Let's Code Some Examples 💻 


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json

# Dataset: World Development Indicators
# Source: World Bank Open Data (data.worldbank.org)
df = pd.read_csv('../data/world_development.csv')
print(f"Loaded: {len(df)} countries")
print(df.columns.tolist())
print(df.head())


Loaded: 20 countries
['Country', 'Region', 'HDI', 'Life_Expectancy', 'GNI_per_capita', 'Fertility_Rate', 'Population_M', 'Urban_pct']
          Country         Region    HDI  Life_Expectancy  GNI_per_capita  \
0          Norway         Europe  0.957             82.4           67614   
1     Switzerland         Europe  0.955             83.8           72051   
2         Germany         Europe  0.942             80.9           56053   
3  United Kingdom         Europe  0.929             81.4           46827   
4   United States  North America  0.921             78.9           65118   

   Fertility_Rate  Population_M  Urban_pct  
0             0.1           5.4         83  
1             0.1           8.6         74  
2             0.3          84.0         77  
3             0.4          67.7         84  
4             0.5         332.0         83  


**Source:** World Bank - 2022 estimates)
**Coverage:** 20 countries, one row per country  

| Column | Description | Units |
|---|---|---|
| `Country` | Country name | — |
| `Region` | Continental grouping (Europe, Africa, Asia, etc.) | — |
| `HDI` | Human Development Index — composite measure of life expectancy, education, and income | 0–1 (higher = more developed) |
| `Life_Expectancy` | Average life expectancy at birth | Years |
| `GNI_per_capita` | Gross National Income per capita — total income earned by residents divided by population | USD |
| `Fertility_Rate` | Average number of children born per woman | Children per woman |
| `Population_M` | Total population | Millions |
| `Urban_pct` | Share of population living in urban areas | % |

### Example 1 — Built-in geometry: world countries (ISO-3)


In [2]:
# ── Exploratory ──────────────────────────────────────────────────────────
# Plotly has built-in world country geometry — just pass ISO-3 codes or country names
# Let's plot Human Developent Index 

fig = px.choropleth(
    df,
    locations='Country',
    locationmode='country names',
    color='HDI',
    title='HDI by Country',
    height=500
)
fig.show()


C:\Users\user\AppData\Local\Temp\ipykernel_11428\651783320.py:5: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


In [3]:
# ── Let's improve it!! ─────────────

fig = px.choropleth(
    df,
    locations='Country',
    locationmode='country names',
    color='HDI',
    color_continuous_scale='Greens',         
    range_color=[0.45, 0.96],               # set range to actual data range
    hover_name='Country',
    hover_data={'HDI': ':.3f', 'Life_Expectancy': ':.1f', 'GNI_per_capita': ':,'},
    labels={'HDI': 'Human Development Index (0–1)'},
    title='Human development divides sharply along geography — Africa lags far behind every other region',
    projection='natural earth'
)
fig.update_layout(
    font=dict(family='Arial', size=12),
    coloraxis_colorbar=dict(title='HDI', thickness=15, len=0.6),
    margin=dict(l=0, r=0, t=55, b=0),
    geo=dict(
        showframe=False      # removes the rectangle border around the map
    )
    
)


fig.show()

C:\Users\user\AppData\Local\Temp\ipykernel_11428\1555208726.py:3: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


### Example 2 — Diverging scale: above/below world average


In [4]:
# ── exploratory: GNI per Capita  ──────────────────────────────────────────────────────────


fig = px.choropleth(
    df,
    locations='Country',
    locationmode='country names',
    color='GNI_per_capita',
    title='GNI per Capita',
    height=500
)
fig.show()

C:\Users\user\AppData\Local\Temp\ipykernel_11428\2820205960.py:4: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


In [5]:
# ── Let's improve it: diverging scale centred on world average ───────────────────

world_avg_gni = df['GNI_per_capita'].mean()
df['GNI_vs_average'] = df['GNI_per_capita'] - world_avg_gni  # positive = above, negative = below

fig = px.choropleth(
    df,
    locations='Country',
    locationmode='country names',
    color='GNI_vs_average',
    color_continuous_scale='RdBu',          # diverging: red = below avg, blue = above
    color_continuous_midpoint=0,            # white at zero = world average
    hover_name='Country',
    hover_data={'GNI_per_capita': ':,', 'GNI_vs_average': ':,.0f', 'Country': False},
    labels={'GNI_vs_average': f'GNI vs World Avg (${world_avg_gni:,.0f})'},
    title=f'Relative wealth: blue = above world average (${world_avg_gni:,.0f}), red = below',
    height=500, projection='natural earth'
)
fig.update_layout(
    font=dict(family='Arial', size=12),
    geo=dict(showframe=False),
    coloraxis_colorbar=dict(title='vs Average\n(USD)', thickness=15, len=0.6),
    margin=dict(l=0, r=0, t=55, b=0),
)
fig.show()



C:\Users\user\AppData\Local\Temp\ipykernel_11428\2543341558.py:6: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


### Example 3 — Scoped map + facets: life expectancy over time (Gapminder)


In [6]:
# Load Gapminder — built into Plotly Express, no file needed
gm = px.data.gapminder()
gm_2007 = gm.loc[gm['year'] == 2007]
print(gm_2007.head())
print(gm_2007.info())


        country continent  year  lifeExp       pop     gdpPercap iso_alpha  \
11  Afghanistan      Asia  2007   43.828  31889923    974.580338       AFG   
23      Albania    Europe  2007   76.423   3600523   5937.029526       ALB   
35      Algeria    Africa  2007   72.301  33333216   6223.367465       DZA   
47       Angola    Africa  2007   42.731  12420476   4797.231267       AGO   
59    Argentina  Americas  2007   75.320  40301927  12779.379640       ARG   

    iso_num  
11        4  
23        8  
35       12  
47       24  
59       32  
<class 'pandas.DataFrame'>
RangeIndex: 142 entries, 11 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    142 non-null    str    
 1   continent  142 non-null    str    
 2   year       142 non-null    int64  
 3   lifeExp    142 non-null    float64
 4   pop        142 non-null    int64  
 5   gdpPercap  142 non-null    float64
 6   iso_alpha  142 non-null    

In [7]:
# ── Scoped map: Europe only ───────────────────────────────────────────────
# scope= zooms the map to a continent; fitbounds= clips to only your data
fig = px.choropleth(
    gm_2007.loc[gm_2007['continent'] == 'Europe'],
    locationmode='ISO-3',
    locations='iso_alpha',
    color='lifeExp',
    hover_name='country',
    color_continuous_scale='Greens',
    #range_color=[70, 85],                   
    scope='europe',                         # zooms map projection to Europe
    fitbounds='locations',                  # clips to only the countries in the data
    labels={'lifeExp': 'Life Expectancy'},
    title='Life expectancy in Europe 2007 — Eastern Europe lags the West by up to 15 years',
    height=400,
)
fig.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=0, r=0, t=55, b=0),
)
fig.show()


In [8]:
# ── Faceted map: compare two years side by side ───────────────────────────
# facet_row= creates one map per year — shared colour scale makes comparison valid
data_two_years = gm.loc[gm['year'].isin([1957, 2007])]

fig = px.choropleth(
    data_two_years,
    locationmode='ISO-3',
    locations='iso_alpha',
    color='lifeExp',
    hover_name='country',
    color_continuous_scale='Greens',
    facet_row='year',                       # one row per year — shared colour scale
    labels={'lifeExp': 'Life Expectancy'},
    title='Life expectancy 1957 vs 2007 — every region improved; gaps narrowed but persist',
    height=600,
)
fig.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=0, r=0, t=55, b=0),
)
fig.show()
# TEACHING POINT: faceted maps share a colour scale automatically.
# This is what makes the comparison valid — both maps encode the same value range.


### Example 4 — Custom GeoJSON: Germany's federal states

When Plotly's built-in geometries don't cover your geography, you need a **GeoJSON file**.

A GeoJSON file:
- Is an open standard (RFC 7946) for encoding geographic shapes as JSON
- Contains a list of **features**, each with a `geometry` (the polygon) and `properties` (metadata like name, ID)
- Is freely available online for most countries, regions, cities, and administrative boundaries

The key argument is `featureidkey` — it tells Plotly which property in the GeoJSON matches your dataframe's `locations` column:
```python
featureidkey='properties.name' 
# means: match df['locations'] to geojson['features'][i]['properties']['name']
```


In [9]:
# Population data for Germany's federal states (2023 estimates)
pop_data = pd.DataFrame({
    'State': [
        'Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen',
        'Hamburg', 'Hessen', 'Niedersachsen', 'Mecklenburg-Vorpommern',
        'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland',
        'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen'
    ],
    'Population': [
        11237000, 13223000, 3769000, 2535200, 680300,
        1858200, 6308900, 7982400, 1610000,
        17947000, 4104000, 984400,
        4051000, 2178000, 2906000, 2118000
    ]
})

# Load the GeoJSON file
with open('../data/4_niedrig.geo.json', 'r') as file:
    germany_geojson = json.load(file)

# Inspect the property key we need for featureidkey
print(germany_geojson['features'][0]['properties'])


FileNotFoundError: [Errno 2] No such file or directory: '../data/4_niedrig.geo.json'

In [ ]:
# ── Step 1: px.choropleth with custom GeoJSON ─────────────────────────────
fig = px.choropleth(
    data_frame=pop_data,
    geojson=germany_geojson,
    locations='State',                      # column in dataframe
    featureidkey='properties.name',         # matching key in GeoJSON properties
    color='Population',
    color_continuous_scale='Oranges',
    fitbounds='locations',                  # auto-zoom to Germany
    hover_name='State',
    hover_data={'State': False},
    labels={'Population': 'Population (2023)'},
    title='Nordrhein-Westfalen and Bayern dominate — eastern states remain the least populated',
    height=400,
)
fig.update_layout(
    font=dict(family='Arial', size=12),
    geo=dict(showframe=False, showcoastlines=False),
    margin=dict(l=0, r=0, t=55, b=0),
)
fig.show()


### Example 5 — `px.choropleth_map`: interactive tile basemap

`px.choropleth_map` adds a Mapbox tile basemap (streets, labels, terrain) instead of Plotly's static projection.

| | `px.choropleth` | `px.choropleth_map` |
|---|---|---|
| Basemap | Static geo projection | Interactive tile map  |
| Zoom/pan | ❌ | ✅ |
| `fitbounds` | ✅ | ❌ — use `center` + `zoom` instead |
| Best for | Static exports, presentations | Dashboards, exploration |


In [ ]:
# ── px.choropleth_map: Germany with interactive basemap ──────────────────
fig = px.choropleth_map(
    data_frame=pop_data,
    geojson=germany_geojson,
    locations='State',
    featureidkey='properties.name',
    color='Population',
    color_continuous_scale='Oranges',
    hover_name='State',
    hover_data={'State': False},
    labels={'Population': 'Population (2023)'},
    title='Population of Germany\'s Federal States',
    center={'lat': 51.1657, 'lon': 10.4515},# Germany center
    zoom=4.8,
    opacity=0.7,                            # show basemap through the polygons
    height=700,
)
fig.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=0, r=0, t=55, b=0),
)
fig.show()

### Example 6 — Sub-district level: Berlin Bezirke

GeoJSON files exist for virtually any administrative boundary — countries, states, cities, postcodes. Here we zoom all the way down to Berlin's 12 districts.


In [ ]:
# Berlin district population data
berlin_pop = pd.DataFrame({
    'district': [
        'Mitte', 'Friedrichshain-Kreuzberg', 'Pankow', 'Charlottenburg-Wilmersdorf',
        'Spandau', 'Steglitz-Zehlendorf', 'Tempelhof-Schöneberg', 'Neukölln',
        'Treptow-Köpenick', 'Marzahn-Hellersdorf', 'Lichtenberg', 'Reinickendorf'
    ],
    'inhabitants': [
        384172, 295864, 415221, 343276,
        245197, 308706, 351622, 335335,
        272843, 271038, 295386, 265755
    ]
})

# Load Berlin GeoJSON
with open('../data/berlin_bezirke.geojson', 'r') as file:
    berlin_geojson = json.load(file)

# Inspect the property key
print(berlin_geojson['features'][0]['properties'])


In [ ]:
# ── px.choropleth_map: Berlin districts ──────────────────────────────────
fig = px.choropleth_map(
    data_frame=berlin_pop,
    geojson=berlin_geojson,
    locations='district',
    featureidkey='properties.name',
    color='inhabitants',
    color_continuous_scale=px.colors.sequential.Blues,
    hover_name='district',
    hover_data={'district': False},
    labels={'inhabitants': 'Estimated Population'},
    title='Pankow is Berlin\'s most populous district — Spandau the least',
    center={'lat': 52.52, 'lon': 13.405},   # Berlin center
    zoom=9.5,
    opacity=0.7,
    height=700,
)
fig.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=0, r=0, t=55, b=0),
)
fig.show()


---
## Class Exercise 💪 💻